In [0]:
import logging
from datetime import datetime

from delta.tables import DeltaTable
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Configure a logger for this ingestion pipeline
logger = logging.getLogger("fr_enrgy_ingestion")
if not logger.hasHandlers():
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(name)s - %(message)s"))
    logger.addHandler(handler)
logger.setLevel(logging.INFO)

logger.info("Starting ingestion pipeline setup")
spark = SparkSession.builder.getOrCreate()
logger.info("Spark session acquired")

target_table = "energy.energy.fr_enrgy"
merge_key = "date_heure"  # must be a TIMESTAMP column in the table (check with DESCRIBE TABLE)

# One entry per source folder, processed one after the other (never in parallel:
# two MERGEs on the same table at once would conflict).
# The first entry keeps your existing checkpoint, so nothing is re-ingested.
SOURCES = [
    {
        "name": "eco2mix-national-tr",
        "path": "/Volumes/energy/energy/energy_volume/eco2mix-national-tr/",
        "checkpoint": "/Volumes/energy/energy/energy_volume/_checkpoints/fr_enrgy/",
    },
    # If your historical DAG writes to another folder, uncomment and adjust.
    # It needs its OWN checkpoint folder.
    # {
    #     "name": "eco2mix-national-cons-def",
    #     "path": "/Volumes/energy/energy/energy_volume/eco2mix-national-cons-def/",
    #     "checkpoint": "/Volumes/energy/energy/energy_volume/_checkpoints/fr_enrgy_history/",
    # },
]


def nature_rank(col_name):
    """Lower = more final. Matches on part of the word so accents/case can't break it."""
    c = F.lower(F.col(col_name))
    return (
        F.when(c.contains("finitive"), 1)   # Données définitives
        .when(c.contains("consolid"), 2)     # Données consolidées
        .otherwise(3)                        # Données temps réel / unknown
    )


# Same ranking as SQL, for the MERGE update condition (t = target, s = source)
NATURE_RANK_SQL = """CASE
    WHEN lower({a}.nature) LIKE '%finitive%' THEN 1
    WHEN lower({a}.nature) LIKE '%consolid%' THEN 2
    ELSE 3 END"""


def process_batch(batch_df, batch_id):
    logger.info("Batch %s: starting processing", batch_id)

    # Unwrap API envelopes like {"results": [...]} / {"records": [...]}, keeping file metadata
    for key in ["results", "records"]:
        if key in batch_df.columns:
            logger.info("Batch %s: unwrapping API envelope under key '%s'", batch_id, key)
            batch_df = batch_df.select(F.explode(key).alias("rec"), "_metadata").select("rec.*", "_metadata")
            break
    # Flatten records nested under "fields" (old Opendatasoft format)
    if "fields" in batch_df.columns:
        logger.info("Batch %s: flattening nested 'fields' struct", batch_id)
        batch_df = batch_df.select("fields.*", "_metadata")

    # File modification time: used only to break ties between copies of the same
    # slot from different files (newest file wins). Not written to the table.
    batch_df = (
        batch_df.withColumn("_file_time", F.col("_metadata.file_modification_time"))
        .drop("_metadata")
    )

    # Drop Auto Loader's rescued-data column if present (not part of the table)
    if "_rescued_data" in batch_df.columns:
        batch_df = batch_df.drop("_rescued_data")

    # Tolerate non-numeric sentinel values (e.g. 'ND' = not available) in the
    # BIGINT columns of the target table: cast them to NULL instead of failing.
    bigint_cols = [
        "ech_comm_allemagne_belgique", "ech_comm_angleterre", "ech_comm_espagne",
        "ech_comm_italie", "ech_comm_suisse", "prevision_j", "prevision_j1",
    ]
    for c in bigint_cols:
        if c in batch_df.columns:
            batch_df = batch_df.withColumn(c, F.expr(f"try_cast(`{c}` AS bigint)"))

    # Rows without a key cannot be merged
    batch_df = batch_df.filter(F.col(merge_key).isNotNull())

    # Deduplicate ON THE KEY: MERGE fails if several source rows match the same
    # target row. Order: most final status first, then rows WITH actual values
    # before forecast-only rows (consommation null), then the newest file.
    # Without the last two, ties between two 'temps réel' copies were random and a
    # forecast-only row (all nulls) could win and overwrite real values.
    logger.info("Batch %s: deduplicating on %s", batch_id, merge_key)
    order_cols = []
    if "nature" in batch_df.columns:
        order_cols.append(nature_rank("nature"))
    if "consommation" in batch_df.columns:
        order_cols.append(F.col("consommation").isNull())  # False (has values) sorts first
    order_cols.append(F.col("_file_time").desc())
    w = Window.partitionBy(merge_key).orderBy(*order_cols)
    deduped = (
        batch_df.withColumn("_rn", F.row_number().over(w))
        .filter("_rn = 1")
        .drop("_rn", "_file_time")
    )

    # First run: create the table from this batch
    if not spark.catalog.tableExists(target_table):
        logger.info("Batch %s: %s does not exist, creating it", batch_id, target_table)
        deduped.write.format("delta").saveAsTable(target_table)
        logger.info("Batch %s: created %s", batch_id, target_table)
        return

    # UPSERT on date_heure:
    #  - existing slot -> update only if the incoming version is at least as final
    #    AND it doesn't replace actual values with an empty forecast row
    #  - new slot      -> insert
    logger.info("Batch %s: merging into %s on %s", batch_id, target_table, merge_key)
    target = DeltaTable.forName(spark, target_table)
    conditions = []
    if "nature" in deduped.columns:
        conditions.append(f"({NATURE_RANK_SQL.format(a='s')}) <= ({NATURE_RANK_SQL.format(a='t')})")
    if "consommation" in deduped.columns:
        conditions.append("NOT (s.consommation IS NULL AND t.consommation IS NOT NULL)")
    update_condition = " AND ".join(conditions) if conditions else None
    (
        target.alias("t")
        .merge(deduped.alias("s"), f"t.{merge_key} = s.{merge_key}")
        .whenMatchedUpdateAll(condition=update_condition)
        .whenNotMatchedInsertAll()
        .execute()
    )

    # Report what the MERGE did. These metrics replace the earlier count() calls:
    # each count() re-read the batch's files, and serverless compute can't cache them.
    metrics = target.history(1).select("operationMetrics").first()[0] or {}
    logger.info(
        "Batch %s: MERGE done - %s source rows, %s inserted, %s updated into %s at %s",
        batch_id,
        metrics.get("numSourceRows", "?"),
        metrics.get("numTargetRowsInserted", "?"),
        metrics.get("numTargetRowsUpdated", "?"),
        target_table,
        datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    )


# Auto Loader: incrementally ingests every NEW file dropped into each folder.
# Note: expects line-delimited JSON (JSON Lines); for files that are one big
# JSON array/object, add .option("multiLine", "true").
# trigger(availableNow=True) processes all pending new files, then stops.
# Schedule this as a job HOURLY (or trigger it from Airflow after the files land):
# running it every 15 min uses 4x the compute and risks the Free Edition daily quota.
for src in SOURCES:
    logger.info("Starting Auto Loader stream '%s' from %s", src["name"], src["path"])
    try:
        query = (
            spark.readStream.format("cloudFiles")
            .option("cloudFiles.format", "json")
            .option("cloudFiles.inferColumnTypes", "true")
            .option("cloudFiles.schemaLocation", f"{src['checkpoint']}schema")
            .load(src["path"])
            .select("*", "_metadata")  # file path + modification time, for tie-breaking
            .writeStream
            .foreachBatch(process_batch)
            .option("checkpointLocation", f"{src['checkpoint']}checkpoint")
            .trigger(availableNow=True)
            .start()
        )
        query.awaitTermination()
        logger.info("Stream '%s' terminated: all pending files processed", src["name"])
    except Exception:
        logger.exception("Ingestion pipeline failed on source '%s'", src["name"])
        raise

2026-11-18 16:53:55,613 INFO fr_enrgy_ingestion - Starting ingestion pipeline setup
2026-11-18 16:53:55,616 INFO fr_enrgy_ingestion - Spark session acquired
2026-11-18 16:53:55,618 INFO fr_enrgy_ingestion - Starting Auto Loader stream 'eco2mix-national-tr' from /Volumes/energy/energy/energy_volume/eco2mix-national-tr/
2026-11-18 16:53:59,488 INFO fr_enrgy_ingestion - Stream 'eco2mix-national-tr' terminated: all pending files processed


In [0]:
# ONE-OFF BACKFILL — run this cell manually when the table's history must be
# rebuilt (e.g. after a TRUNCATE); the scheduled job does not need it.
# Re-reads ALL files currently in each source volume in batch mode and runs
# them through the same process_batch MERGE as the streaming path. Idempotent
# (dedup + MERGE on date_heure), so re-running is safe. The Auto Loader
# checkpoint is untouched: files it already processed are never re-ingested by
# the stream, which is exactly why this manual backfill is needed after a
# truncate.
for src in SOURCES:
    logger.info("Backfill '%s': reading all files from %s", src["name"], src["path"])
    backfill_df = (
        spark.read.format("json")
        .option("samplingRatio", "1.0")  # infer schema from ALL files, not a sample
        .load(src["path"])
        .select("*", "_metadata")
    )
    process_batch(backfill_df, "backfill")
    logger.info("Backfill '%s': complete", src["name"])